# Konu 2 — İnterneti koddan konuşturmak

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

## Isınma — iç içe sözlük

Modelden gelen yanıt **sözlüğün içinde sözlük** olacak. Önce elle bir tane kuralım.
Aşağıdaki son satır hata veriyor. Neden?

In [ ]:
yanit = {
    "success": True,
    "result": {"response": "Merhaba!"},
}

print(yanit["response"])

`"response"` doğrudan `yanit`'ın içinde değil, `"result"`'ın içinde. Katman katman inelim:

In [ ]:
sonuc = yanit["result"]
print(sonuc)

In [ ]:
metin = sonuc["response"]
print(metin)

## Adım 1 — Anahtarı dosyadan oku

Anahtar kodun içine yazılmaz; `gita3111` klasöründeki `anahtar.txt` dosyasında durur.
`..` "bir üst klasör" demek.

In [ ]:
anahtarlar = {}
with open("../anahtar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        parcalar = satir.split("=")
        ad = parcalar[0].strip()
        deger = parcalar[1].strip()
        anahtarlar[ad] = deger

hesap = anahtarlar["ACCOUNT_ID"]
anahtar = anahtarlar["API_TOKEN"]
print(hesap)

## Adım 2 — İsteği hazırla: adres, başlık, gövde

- **Adres:** hangi modele gidiyoruz
- **Başlık:** ben kimim (anahtar burada gider)
- **Gövde:** ne istiyorum

In [ ]:
import requests

MODEL = "@cf/google/gemma-4-26b-a4b-it"
adres = f"https://api.cloudflare.com/client/v4/accounts/{hesap}/ai/run/{MODEL}"
basliklar = {"Authorization": f"Bearer {anahtar}"}
govde = {"prompt": "Sessiz bir çalışma kafesi için üç kısa slogan yaz."}

## Adım 3 — Gönder

`200` "tamam" demek.

In [ ]:
cevap = requests.post(adres, headers=basliklar, json=govde)
print(cevap.status_code)

## Adım 4 — Gelen yanıta bak

Gelen yanıt bir sözlük. Isınmadakinin aynısı, sadece daha kalabalık.

In [ ]:
yanit = cevap.json()
print(yanit)

In [ ]:
sonuc = yanit["result"]
metin = sonuc["response"]
print(metin)

## Adım 5 — Yanlış anahtarla dene

Bilerek sahte bir anahtar gönderiyoruz. Durum kodu ne oldu?

In [ ]:
sahte_basliklar = {"Authorization": "Bearer yanlis-anahtar"}
cevap = requests.post(adres, headers=sahte_basliklar, json=govde)
print(cevap.status_code)
print(cevap.text)

Yanıta güvenmeden önce durum koduna bakarız:

In [ ]:
if cevap.status_code == 200:
    print("Tamam")
else:
    print("Bir sorun var:", cevap.status_code)

## Adım 6 — Fonksiyona koy

Her soru için aynı dört satırı yazmamak için bir fonksiyon yapıyoruz.

In [ ]:
def modele_sor(soru):
    govde = {"prompt": soru}
    cevap = requests.post(adres, headers=basliklar, json=govde)
    yanit = cevap.json()
    sonuc = yanit["result"]
    return sonuc["response"]

In [ ]:
print(modele_sor("Sessiz bir çalışma kafesinin logosu için üç renk öner."))

## Adım 7 — Birden çok soru, cevaplar dosyaya

In [ ]:
sorular = [
    "Sessiz bir çalışma kafesinin logosu için hangi üç rengi önerirsin?",
    "Bu kafenin afişinde serif mi sans-serif mi yazı tipi uygun olur?",
    "Bu kafenin Instagram hesabı için üç gönderi fikri ver.",
]

with open("cevaplar.txt", "w", encoding="utf-8") as dosya:
    for soru in sorular:
        metin = modele_sor(soru)
        print(soru)
        print(metin)
        dosya.write(soru + "\n" + metin + "\n\n")

## Adım 8 — Soruya bağlam koymak cevabı değiştirir mi?

Konu 1'de müşterilerin bu kafeyi **sessiz bir çalışma yeri** olarak anlattığını bulmuştuk.
Bu bulguyu soruya koyunca sloganlar nasıl değişiyor?

In [ ]:
baglamsiz = modele_sor("Bir kafe için üç kısa slogan yaz.")
baglamli = modele_sor("Müşterilerinin sessiz bir çalışma yeri olarak anlattığı bir kafe için üç kısa slogan yaz.")

print(baglamsiz)
print()
print(baglamli)

İki cevabı yan yana oku. Hangisinde "kahve" var, hangisinde "sessizlik", "odak"?
Bağlamsız soruda model **ortalama bir kafeyi** anlatır. Soruyu yazmak bir tasarım kararıdır.